In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("GPU is not available.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU name: Tesla T4
GPU memory: 14.56 GB


In [4]:
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas numpy matplotlib seaborn

In [5]:
import torch
import transformers
import datasets
import evaluate
import accelerate
import sklearn
import pandas as pd
import numpy as np

print("All libraries loaded successfully.")
print()
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("Accelerate:", accelerate.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print()
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


All libraries loaded successfully.

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
Datasets: 4.0.0
Evaluate: 0.4.6
Accelerate: 1.14.0
Scikit-learn: 1.6.1
Pandas: 2.2.3
NumPy: 2.1.3

CUDA available: True
GPU: Tesla T4


In [6]:
from google.colab import files

uploaded = files.upload()

Saving ethereum_test.csv to ethereum_test.csv
Saving ethereum_train.csv to ethereum_train.csv
Saving ethereum_validation.csv to ethereum_validation.csv


In [7]:
import pandas as pd
from pathlib import Path

train = pd.read_csv("ethereum_train.csv")
validation = pd.read_csv("ethereum_validation.csv")
test = pd.read_csv("ethereum_test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print("\nColumns:")
print(train.columns.tolist())

print("\nLabel values:")
print("Train:", sorted(train["labels"].unique()))
print("Validation:", sorted(validation["labels"].unique()))
print("Test:", sorted(test["labels"].unique()))

Train: (3794, 13)
Validation: (470, 13)
Test: (457, 13)

Columns:
['timestamp', 'title', 'description', 'text', 'market_direction', 'engagement_quality', 'content_characteristics', 'vote_counts', 'total_votes', 'source_url', 'url', 'total_tokens', 'labels']

Label values:
Train: [np.int64(0), np.int64(1), np.int64(2)]
Validation: [np.int64(0), np.int64(1), np.int64(2)]
Test: [np.int64(0), np.int64(1), np.int64(2)]


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "ProsusAI/finbert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

text = test.iloc[0]["text"]

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)
predicted_class = torch.argmax(probabilities, dim=-1).item()

print("Model loaded successfully.")
print("Device:", device)
print("Original FinBERT labels:", model.config.id2label)
print("Predicted class ID:", predicted_class)
print("Probabilities:", probabilities[0].cpu().tolist())

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully.
Device: cuda
Original FinBERT labels: {0: 'positive', 1: 'negative', 2: 'neutral'}
Predicted class ID: 2
Probabilities: [0.14136521518230438, 0.014349843375384808, 0.8442848920822144]


In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import torch

texts = test["text"].tolist()
true_labels = test["labels"].tolist()

predicted_labels = []

batch_size = 16

for start in range(0, len(texts), batch_size):
    batch_texts = texts[start:start + batch_size]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    batch_predictions = torch.argmax(outputs.logits, dim=-1)
    predicted_labels.extend(batch_predictions.cpu().tolist())

accuracy = accuracy_score(true_labels, predicted_labels)

print("Baseline accuracy:", round(accuracy, 4))
print("\nClassification report:")
print(
    classification_report(
        true_labels,
        predicted_labels,
        labels=[0, 1, 2],
        target_names=["bullish", "bearish", "neutral"],
        digits=4
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        true_labels,
        predicted_labels,
        labels=[0, 1, 2]
    )
)

Baseline accuracy: 0.4004

Classification report:
              precision    recall  f1-score   support

     bullish     0.3483    0.3899    0.3680       159
     bearish     0.5368    0.4722    0.5025       108
     neutral     0.3804    0.3684    0.3743       190

    accuracy                         0.4004       457
   macro avg     0.4219    0.4102    0.4149       457
weighted avg     0.4062    0.4004    0.4024       457

Confusion matrix:
[[62 19 78]
 [21 51 36]
 [95 25 70]]


In [10]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

# Keep only the columns required for model training.
train_data = Dataset.from_pandas(
    train[["text", "labels"]],
    preserve_index=False
)

validation_data = Dataset.from_pandas(
    validation[["text", "labels"]],
    preserve_index=False
)

test_data = Dataset.from_pandas(
    test[["text", "labels"]],
    preserve_index=False
)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

tokenized_train = train_data.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

tokenized_validation = validation_data.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

tokenized_test = test_data.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Tokenization completed.")
print("Train examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_validation))
print("Test examples:", len(tokenized_test))
print("Tokenized columns:", tokenized_train.column_names)

Map:   0%|          | 0/3794 [00:00<?, ? examples/s]

Map:   0%|          | 0/470 [00:00<?, ? examples/s]

Map:   0%|          | 0/457 [00:00<?, ? examples/s]

Tokenization completed.
Train examples: 3794
Validation examples: 470
Test examples: 457
Tokenized columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [11]:
from transformers import AutoModelForSequenceClassification

id2label = {
    0: "bullish",
    1: "bearish",
    2: "neutral"
}

label2id = {
    "bullish": 0,
    "bearish": 1,
    "neutral": 2
}

fine_tuned_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

fine_tuned_model.to(device)

print("Fine-tuning model loaded successfully.")
print("Device:", device)
print("Labels:", fine_tuned_model.config.id2label)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Fine-tuning model loaded successfully.
Device: cuda
Labels: {0: 'bullish', 1: 'bearish', 2: 'neutral'}


In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

training_args = TrainingArguments(
    output_dir="./ethereum-finbert-checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    report_to="none",
    seed=42,
    fp16=True
)

print("Training arguments configured.")
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Training batch size:", training_args.per_device_train_batch_size)

Training arguments configured.
Epochs: 3
Learning rate: 2e-05
Training batch size: 16


In [13]:
from transformers import Trainer

trainer = Trainer(
    model=fine_tuned_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created successfully.")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))

Trainer created successfully.
Training examples: 3794
Validation examples: 470


In [14]:
training_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.009592,0.967660,0.487234,0.539587,0.483800,0.462164
2,0.834847,0.955036,0.527660,0.552844,0.522198,0.532997
3,0.704155,0.987744,0.536170,0.554704,0.531521,0.539727


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

test_prediction = trainer.predict(tokenized_test)

test_logits = test_prediction.predictions
test_true_labels = test_prediction.label_ids
test_predicted_labels = test_logits.argmax(axis=-1)

test_accuracy = accuracy_score(
    test_true_labels,
    test_predicted_labels
)

print("Fine-tuned test accuracy:", round(test_accuracy, 4))

print("\nFine-tuned classification report:")
print(
    classification_report(
        test_true_labels,
        test_predicted_labels,
        labels=[0, 1, 2],
        target_names=["bullish", "bearish", "neutral"],
        digits=4
    )
)

print("Fine-tuned confusion matrix:")
print(
    confusion_matrix(
        test_true_labels,
        test_predicted_labels,
        labels=[0, 1, 2]
    )
)

Fine-tuned test accuracy: 0.5689

Fine-tuned classification report:
              precision    recall  f1-score   support

     bullish     0.5405    0.5031    0.5212       159
     bearish     0.6436    0.6019    0.6220       108
     neutral     0.5529    0.6053    0.5779       190

    accuracy                         0.5689       457
   macro avg     0.5790    0.5701    0.5737       457
weighted avg     0.5700    0.5689    0.5686       457

Fine-tuned confusion matrix:
[[ 80  14  65]
 [ 15  65  28]
 [ 53  22 115]]


In [16]:
import pandas as pd

comparison = pd.DataFrame({
    "metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ],
    "baseline": [
        0.4004,
        0.4219,
        0.4102,
        0.4149,
        0.4024
    ],
    "fine_tuned": [
        0.5689,
        0.5790,
        0.5701,
        0.5737,
        0.5686
    ]
})

comparison["improvement"] = (
    comparison["fine_tuned"] - comparison["baseline"]
)

comparison

,metric,baseline,fine_tuned,improvement
0,Accuracy,0.4004,0.5689,0.1685
1,Macro Precision,0.4219,0.5790,0.1571
2,Macro Recall,0.4102,0.5701,0.1599
3,Macro F1,0.4149,0.5737,0.1588
4,Weighted F1,0.4024,0.5686,0.1662


In [17]:
comparison.to_csv(
    "model_comparison.csv",
    index=False
)

print("Comparison saved to model_comparison.csv")

Comparison saved to model_comparison.csv


In [18]:
from google.colab import files

files.download("model_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from pathlib import Path

final_model_dir = Path("/content/ethereum-finbert-final")
final_model_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

print("Model and tokenizer saved to:")
print(final_model_dir)
print("Saved files:")
print(sorted(path.name for path in final_model_dir.iterdir()))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to:
/content/ethereum-finbert-final
Saved files:
['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [20]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

final_model_dir = Path("/content/ethereum-finbert-final")

final_metrics = {
    "accuracy": float(test_accuracy),
    "classification_report": classification_report(
        test_true_labels,
        test_predicted_labels,
        labels=[0, 1, 2],
        target_names=["bullish", "bearish", "neutral"],
        output_dict=True
    ),
    "confusion_matrix": confusion_matrix(
        test_true_labels,
        test_predicted_labels,
        labels=[0, 1, 2]
    ).tolist(),
    "label_mapping": {
        "0": "bullish",
        "1": "bearish",
        "2": "neutral"
    }
}

with open(final_model_dir / "test_metrics.json", "w") as file:
    json.dump(final_metrics, file, indent=2)

print("Test metrics saved.")

Test metrics saved.
